# 02 - Motion Flow: animating a picture by warping its pixels

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/matu1003/Makeitalive/blob/main/notebooks/02_motion_flow.ipynb)

The naive approach: a lightweight U-Net predicts a dense flow field `(dx, dy)` from a **single image**,
and the picture is animated by moving its own pixels along this flow.

- It is fast and needs no large pretrained model.
- It cannot create new content: occluded regions and camera motion (parallax) are out of reach.

This notebook goes from the intuition to training and inference. All the logic lives in `src/motion_flow/`.

In [ ]:
import sys
from pathlib import Path

# On Colab, clone the repo and install it; locally, run `uv sync` first.
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone -q https://github.com/matu1003/Makeitalive.git
    %cd Makeitalive
    !pip install -q -e .
    REPO_ROOT = Path.cwd()
else:
    REPO_ROOT = Path.cwd().parent

DATA_DIR = REPO_ROOT / "data"
CKPT_DIR = REPO_ROOT / "checkpoints"
OUTPUT_DIR = REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

import numpy as np
import torch
import matplotlib.pyplot as plt
from types import SimpleNamespace
from IPython.display import Image as ShowImage

from motion_flow.model import MotionFlowUNet
from motion_flow.warp import warp
from motion_flow import train as mf_train
from motion_flow.infer import (
    find_latest_checkpoint, load_model, load_image, predict_flow,
    animate, animate_autoregressive, save_animation, to_uint8,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_PATH = REPO_ROOT / "assets" / "images" / "landscape.jpg"
print(f"Device: {device}")

## 1. Intuition: a hand-made flow field

Warping an image with a flow means `output(x) = input(x + flow(x))`. With a synthetic wave-shaped flow and an
amplitude that grows over time, a static picture already looks alive. The model's job is to predict a *plausible*
flow instead of this hand-made one.

In [ ]:
img = load_image(str(IMAGE_PATH), size=512)
_, _, H, W = img.shape

yy, xx = torch.meshgrid(torch.arange(H), torch.arange(W), indexing="ij")
wave = torch.stack([torch.sin(yy * 0.02), torch.cos(xx * 0.02)]).float().unsqueeze(0)  # (1, 2, H, W)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(to_uint8(img)); axes[0].set_title("Input")
axes[1].imshow(wave[0, 0], cmap="coolwarm"); axes[1].set_title("Synthetic flow dx")
axes[2].imshow(wave[0, 1], cmap="coolwarm"); axes[2].set_title("Synthetic flow dy")
axes[3].imshow(to_uint8(warp(img, wave * 15))); axes[3].set_title("Warped (amplitude 15 px)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

save_animation(animate(img, wave, num_frames=30, magnitude=15, ping_pong=True), str(OUTPUT_DIR / "synthetic_wave.gif"), fps=15)
ShowImage(filename=str(OUTPUT_DIR / "synthetic_wave.gif"))

## 2. Model: a lightweight U-Net

Input: one RGB image `(3, H, W)`. Output: a flow field `(2, H, W)` in pixels.
Channel widths are halved compared to the original U-Net (32 to 512) so that 512x512 images fit in GPU memory.

In [ ]:
model = MotionFlowUNet()
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params / 1e6:.1f} M")
print(f"Input {tuple(img.shape)} -> flow {tuple(model(img).shape)}")

## 3. Self-supervised training

No ground-truth flow is available, so the flow is learned through reconstruction:

$$\mathcal{L} = \big\| \, \text{warp}(I_t, f_\theta(I_t)) - I_{t+k} \, \big\|_2^2$$

The model predicts a flow from `I_t` alone, `I_t` is warped with it, and the result must match the real next frame `I_{t+k}`.
The pairs come from `01_dataset.ipynb`. Mixed precision is used on GPU only.
Checkpoints (`model_best.pth`, `model_latest.pth`) are written to `checkpoints/run_<timestamp>/`.

In [ ]:
TRAIN = False  # set to True to train (a GPU is strongly recommended)

if TRAIN:
    mf_train.train(SimpleNamespace(
        data_dir=str(DATA_DIR / "dataset_local"),
        ckpt_dir=str(CKPT_DIR),
        epochs=50,
        batch_size=8,
        lr=1e-4,
        num_workers=4,
    ))

## 4. Inference: predicted flow and animation

The trained model predicts one flow field for the picture. The animation warps the image with this flow,
scaled from 0 to `magnitude` pixels over the frames.

No pretrained weights are shipped with the repo: without a checkpoint in `checkpoints/`, the model keeps random weights, so run the training cell above first (`TRAIN = True`).

In [ ]:
checkpoint = find_latest_checkpoint(str(CKPT_DIR))
print(f"Checkpoint: {checkpoint}" if checkpoint else "No checkpoint found: using random weights, train the model first.")
model = load_model(checkpoint, device)

flow = predict_flow(model, img)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(to_uint8(img)); axes[0].set_title("Input")
for ax, i, title in zip(axes[1:], [0, 1], ["Predicted dx", "Predicted dy"]):
    im = ax.imshow(flow[0, i], cmap="coolwarm")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
frames = animate(img, flow, num_frames=60, magnitude=25, ping_pong=True)
save_animation(frames, str(OUTPUT_DIR / "motion_flow.gif"), fps=20)
ShowImage(filename=str(OUTPUT_DIR / "motion_flow.gif"))

## 5. Autoregressive variant (experimental)

Instead of scaling a single flow, the flow is re-predicted on each generated frame. The motion can evolve over time,
but every resampling step blurs the picture a little more.

In [ ]:
frames = animate_autoregressive(model, img, steps=30, magnitude=0.5)
save_animation(frames, str(OUTPUT_DIR / "motion_flow_autoregressive.gif"), fps=10)
ShowImage(filename=str(OUTPUT_DIR / "motion_flow_autoregressive.gif"))

## 6. Results and limitations

Results of the trained model:

| Wheat | Lake |
|:---:|:---:|
| <img src="../assets/gifs/motion_flow_wheat.gif" width="320"> | <img src="../assets/gifs/motion_flow_lake.gif" width="320"> |

Textures such as wheat or water move convincingly, but warping only **moves existing pixels**:
no new content appears, so the camera cannot really move through the scene. This motivates the generative
approach of `03_svd_lora.ipynb`.